# TikTok Shop — ทดลองการทำงานของ Data Pipeline

**เปิดใน Google Colab แล้วเลือก Runtime → Run all ครั้งเดียว** ระบบจะทำงานจากบนลงล่าง และแสดงผลแยกใต้แต่ละขั้น เพื่อให้เห็นว่าข้อมูลถูกนำเข้า จัดเก็บ แปลง และนำไปใช้วิเคราะห์อย่างไร

**ข้อมูลใน Demo นี้เป็นข้อมูลการขายจำลองของร้านเสื้อผ้า ไม่ได้ดึงจาก TikTok Shop จริง** ยอดขาย ต้นทุน และค่าธรรมเนียมเป็นตัวเลขตามสมมติฐานของโครงการ

| คำที่เห็นใน Demo | หมายถึงอะไร | ใช้ทำอะไร |
|---|---|---|
| **Source Data (CSV)** | ไฟล์ข้อมูลตั้งต้นบน GitHub | ข้อมูลที่ระบบจะรับเข้ามา |
| **ER1 — SQLite Database** | ฐานข้อมูลต้นทางตามแบบ ER1 | เก็บคำสั่งซื้อและข้อมูลธุรกรรมเป็นตารางที่เชื่อมโยงกัน |
| **Data Lake — Parquet** | พื้นที่เก็บไฟล์ระหว่างทาง **ไม่ใช่ ER1** | เก็บข้อมูลที่รับเข้า ข้อมูลดิบจาก ER1 และข้อมูลที่แปลงแล้ว |
| **ER2 — DuckDB Data Warehouse** | คลังข้อมูลตามแบบ ER2 | จัดข้อมูลให้เหมาะกับการวิเคราะห์ยอดขาย ต้นทุน และกำไร |
| **Reports** | รายงานที่คำนวณจาก ER2 และผลตรวจสอบ | ใช้อ่านผลธุรกิจและตรวจความถูกต้องของ Pipeline |

**การทดลองมี 4 ช่วง:** Run 1 นำข้อมูลตั้งต้นเข้าทุกชั้น → Run 2 เพิ่ม/อัปเดตข้อมูลในฐานเดิม → Replay ส่ง Batch เดิมซ้ำเพื่อพิสูจน์ว่าไม่เพิ่มยอดซ้ำ → จำลอง Error แล้วตรวจการกู้คืน

> ทุกครั้งที่กด Run all ใหม่ ระบบล้างเฉพาะผลทดลองใน Colab `output/main` **ก่อน Run 1 เพียงครั้งเดียว** ไม่ได้ล้างข้อมูลระหว่าง Run 1 กับ Run 2 และไม่ลบไฟล์ต้นทางบน GitHub หรือไฟล์ใน Google Drive


## 00 — เตรียมพื้นที่ทดลอง

**กำลังทำอะไร:** โหลด Code และไฟล์ข้อมูลจำลองจาก GitHub มายัง Colab พร้อมเตรียมพื้นที่เก็บผลลัพธ์

**ผลที่เห็น:** ตำแหน่งโปรเจกต์ จำนวนออเดอร์ในไฟล์เริ่มต้นและไฟล์รอบใหม่ รวมถึงโฟลเดอร์ที่เก็บ Database, Data Lake และรายงาน

**หมายเหตุ:** ขั้นนี้ยังไม่ได้นำ CSV เข้าฐานข้อมูลค่ะ

In [ ]:
#@title 00 — เตรียมโปรเจกต์
REPO_URL = "https://github.com/KankaweeK356/tiktok-shop-pipeline-demo"
from pathlib import Path
import os, sys, subprocess, importlib, sqlite3
from IPython.display import display, Markdown

project_name = "TikTok_Shop_Pipeline_Demo"
if (Path.cwd() / "pipeline" / "orchestrator.py").is_file():
    ROOT = Path.cwd()
else:
    ROOT = Path("/content") / project_name if Path("/content").is_dir() else Path.cwd() / project_name
    if not ROOT.exists():
        print("กำลังดาวน์โหลด Source และ Code จาก GitHub ...")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
    else:
        print("ใช้ไฟล์โปรเจกต์ที่มีใน Runtime นี้ ไม่ git pull ซ้ำ")
required = ["requirements.txt", "pipeline/orchestrator.py", "pipeline/transform.py",
            "source/initial/Fact_Orders.csv", "source/incremental/Fact_Orders.csv"]
missing = [p for p in required if not (ROOT / p).is_file()]
if missing:
    raise RuntimeError("โครงสร้างโปรเจกต์ไม่ครบ: " + ", ".join(missing))
try:
    import duckdb, pyarrow, pandas, matplotlib, nbformat
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                           str(ROOT / "requirements.txt")])
root_path = str(ROOT.resolve())
sys.path = [p for p in sys.path if p != root_path]
sys.path.insert(0, root_path)
for module_name in list(sys.modules):
    if module_name == "pipeline" or module_name.startswith("pipeline."):
        del sys.modules[module_name]
importlib.invalidate_caches()

from pipeline.orchestrator import run_batch, reset
from pipeline.demo_tests import signature, error_suite, performance_suite
from pipeline.quality import verify, lineage
from pipeline.business import business_report
from pipeline.publication import export_github_reports, EvidenceNotReady
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

WORKSPACE = ROOT / "output" / "main"
# Run all always starts a fresh, reproducible demo. The reset runs ONCE
# here, before Run 1. Run 2 updates the same ER1/ER2 databases; no reset
# happens between batches. Never touch GitHub source or Drive snapshots.
reset(WORKSPACE)
print("✅ เริ่ม Demo ใหม่ — ล้างเฉพาะผลเก่า output/main ใน Colab")
print("Run 1 และ Run 2 จะใช้ Database เดิมต่อเนื่องกัน ไม่มีการ Reset คั่น")
print("✅ พร้อมทดลอง | โปรเจกต์:", ROOT)
print("Source เริ่มต้น:", len(pd.read_csv(ROOT/"source/initial/Fact_Orders.csv")), "Orders")
print("Source เพิ่มเติม:", len(pd.read_csv(ROOT/"source/incremental/Fact_Orders.csv")), "แถวคำสั่งซื้อ")
print("ผลลัพธ์จะอยู่ที่:", WORKSPACE)

def explain_result(text):
    """Short human explanation shown beneath the actual stage output."""
    display(Markdown("**อ่านผลตรงนี้:** " + str(text)))

explain_result("ตอนนี้เพียงเตรียมไฟล์และพื้นที่ไว้ก่อน ยังไม่มีการนำข้อมูลเข้า ER1 หรือ ER2\nเมื่อ Run all ทำงานต่อ ขั้นถัดไปจะเปิดไฟล์ CSV ให้ดู")


## RUN 1 — นำข้อมูลรอบแรกเข้าระบบ

เราจะเริ่มจากไฟล์ CSV แล้วค่อย ๆ ดูข้อมูลที่เกิดขึ้นใน **ฐานข้อมูล ER1 → Data Lake → คลังข้อมูล ER2** โดยใช้ข้อมูลชุดตั้งต้นเดียวกันทั้งเส้นทาง

### 01 — ดูข้อมูลต้นทาง (Source CSV)

**กำลังทำอะไร:** เปิดไฟล์คำสั่งซื้อและรายการสินค้าเพื่อดูว่าก่อนนำเข้าระบบมีข้อมูลอยู่เท่าไร และหน้าตาข้อมูลเป็นแบบไหน

**ผลที่ควรดู:** จำนวนไฟล์ CSV จำนวนออเดอร์ จำนวนรายการสินค้าในออเดอร์ และตัวอย่างข้อมูล 5 แถว

**ทำไมต้องดู:** เราจะใช้จำนวนเหล่านี้เป็นจุดเริ่มต้นสำหรับเทียบกับข้อมูลหลังนำเข้า

In [ ]:
#@title 01 — ตรวจข้อมูล CSV ก่อนนำเข้า
initial_dir = ROOT / "source" / "initial"
source_orders = pd.read_csv(initial_dir / "Fact_Orders.csv")
source_items = pd.read_csv(initial_dir / "Fact_Order_Items.csv")
source_files = list(initial_dir.glob("*.csv"))

print("ไฟล์ Source CSV ทั้งหมด:", len(source_files), "ไฟล์")
display(pd.DataFrame([
    {"ข้อมูลในไฟล์": "Fact_Orders.csv", "หมายถึง": "คำสั่งซื้อ 1 แถวต่อออเดอร์", "จำนวนแถว": len(source_orders)},
    {"ข้อมูลในไฟล์": "Fact_Order_Items.csv", "หมายถึง": "สินค้าที่อยู่ในคำสั่งซื้อ (1 ออเดอร์มีได้หลายรายการ)", "จำนวนแถว": len(source_items)},
]))
print("ตัวอย่างคำสั่งซื้อจากไฟล์ CSV (5 แถว):")
display(source_orders[["Order_ID", "Current_Status", "Paid_At"]].head(5).rename(columns={
    "Order_ID": "เลขที่ออเดอร์", "Current_Status": "สถานะคำสั่งซื้อ",
    "Paid_At": "วันเวลาชำระเงิน"
}))
explain_result("นี่คือข้อมูลต้นทางที่ยังเป็น CSV ไม่ใช่ข้อมูลใน Database จำนวนออเดอร์กับจำนวนรายการสินค้าอาจไม่เท่ากัน เพราะหนึ่งออเดอร์ซื้อได้หลายรายการ")


### 02 — นำ CSV เข้าฐานข้อมูลต้นทาง (SQLite — ER1)

**ER1 คืออะไร:** ER1 เป็น **แบบโครงสร้างฐานข้อมูลต้นทาง** ใน Demo นี้เราใช้ SQLite สร้างตารางตามแบบ ER1 เพื่อเก็บข้อมูลคำสั่งซื้อ รายการสินค้าที่ขาย และข้อมูลสินค้าให้เชื่อมโยงกันได้

**กำลังทำอะไร:** นำข้อมูล CSV เข้าตารางใน SQLite แล้วนับ Records ที่บันทึกจริง

**ผลที่ควรดู:** จำนวนตารางทั้งหมด จำนวน Records ในตัวอย่าง 3 ตาราง และข้อมูลตัวอย่างที่อ่านกลับจากฐานข้อมูล

**สำคัญ:** ER1 เป็นส่วนของ **Database** ไม่ใช่ Data Lake

In [ ]:
#@title 02 — นำ Source CSV เข้าฐานข้อมูล SQLite (ER1)
stage = run_batch(WORKSPACE, "initial", stop_after="ingest")
with sqlite3.connect(WORKSPACE / "er1.sqlite") as con:
    er1_table_count = con.execute(
        "SELECT COUNT(*) FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    ).fetchone()[0]
    er1_counts = pd.read_sql_query("""SELECT 'Fact_Orders' AS Table_Name, COUNT(*) AS Records FROM Fact_Orders
      UNION ALL SELECT 'Fact_Order_Items',COUNT(*) FROM Fact_Order_Items
      UNION ALL SELECT 'Dim_SKU',COUNT(*) FROM Dim_SKU""", con)
    er1_example = pd.read_sql_query(
        "SELECT Order_ID,Current_Status,Paid_At FROM Fact_Orders LIMIT 5", con)
er1_explanation = {
    "Fact_Orders": "คำสั่งซื้อและสถานะของออเดอร์",
    "Fact_Order_Items": "สินค้าที่อยู่ในแต่ละออเดอร์",
    "Dim_SKU": "รายละเอียดรหัสสินค้า"
}
print("ฐานข้อมูลต้นทาง: SQLite (ออกแบบตาม ER1)")
print("จำนวนตารางทั้งหมดใน ER1:", er1_table_count, "ตาราง")
er1_counts["เก็บข้อมูลอะไร"] = er1_counts["Table_Name"].map(er1_explanation)
display(er1_counts.rename(columns={"Table_Name": "ชื่อตาราง", "Records": "จำนวนแถว"}))
print("ตัวอย่างออเดอร์ที่ Query กลับจาก SQLite:")
display(er1_example.rename(columns={"Order_ID":"เลขที่ออเดอร์",
    "Current_Status":"สถานะ","Paid_At":"ชำระเงินเมื่อ"}))
assert er1_counts.loc[er1_counts.Table_Name=="Fact_Orders","Records"].iloc[0] == len(source_orders)
explain_result("ไฟล์ CSV ถูกนำเข้าตารางของ ER1 แล้ว ขั้นต่อไปจะอ่านข้อมูลจาก SQLite ไปเก็บเป็น Raw Parquet ใน Data Lake โดย ER1 ไม่ใช่ Data Lake")


### 03 — เก็บข้อมูลใน Data Lake: Landing และ Raw

**Data Lake คืออะไร:** เป็นพื้นที่เก็บไฟล์ข้อมูลระหว่างการประมวลผล ใน Demo นี้ใช้ไฟล์ **CSV และ Parquet** ไม่ใช่ฐานข้อมูล ER1 หรือ ER2

**กำลังทำอะไร:** เก็บสำเนา CSV ที่ได้รับไว้ใน **Landing** แล้วอ่านข้อมูลจาก SQLite (ER1) มาบันทึกเป็นไฟล์ **Raw Parquet** เพื่อให้ตรวจย้อนหลังได้

**ผลที่ควรดู:** จำนวนไฟล์ Landing/Raw จำนวนแถวของ Raw Orders และตัวอย่างข้อมูลที่อ่านจาก Parquet

**ตอนนี้ยังไม่ได้จัดข้อมูลให้พร้อมวิเคราะห์ใน Warehouse**

In [ ]:
#@title 03 — เก็บสำเนา CSV และข้อมูล Raw ใน Data Lake
stage = run_batch(WORKSPACE, "initial", stop_after="extract")
landing = WORKSPACE / "lake" / "landing" / "initial"
raw = WORKSPACE / "lake" / "raw" / "initial"
raw_orders = pd.read_parquet(raw / "Fact_Orders.parquet")
landing_files = list(landing.glob("*.csv"))
raw_files = list(raw.glob("*.parquet"))
print("Data Lake: ที่เก็บไฟล์ข้อมูลระหว่างทาง")
display(pd.DataFrame([
    {"พื้นที่": "Landing", "เก็บข้อมูลอะไร": "สำเนา Source CSV ที่รับเข้า", "จำนวนไฟล์": len(landing_files)},
    {"พื้นที่": "Raw", "เก็บข้อมูลอะไร": "ข้อมูลจาก ER1 ที่บันทึกเป็น Parquet", "จำนวนไฟล์": len(raw_files)}
]))
print("Raw Fact_Orders:", len(raw_orders), "แถว | ตัวอย่างที่อ่านจากไฟล์ Parquet:")
display(raw_orders[["Order_ID", "Current_Status", "Paid_At"]].head(5).rename(columns={
    "Order_ID": "เลขที่ออเดอร์", "Current_Status": "สถานะ",
    "Paid_At": "ชำระเงินเมื่อ"
}))
assert len(raw_orders) == len(source_orders)
explain_result("Landing เก็บสำเนาไฟล์ที่รับมา ส่วน Raw เก็บภาพข้อมูลที่อ่านจาก ER1 ก่อนจัดรูปแบบเพื่อวิเคราะห์ จึงย้อนตรวจไฟล์แต่ละช่วงได้")


### 04 — จัดรูปแบบข้อมูลใน Data Lake: Standardized

**กำลังทำอะไร:** แปลงข้อมูลจาก Raw Parquet ให้ตรงกับโครงสร้างที่จะนำเข้า ER2 เช่น จัดรายการขายในระดับ **Order Item** และคำนวณยอดขายสุทธิ ต้นทุนสินค้า และค่าธรรมเนียมตามเงื่อนไขที่กำหนด

**ผลที่ควรดู:** จำนวนไฟล์ Standardized จำนวนรายการขายที่เตรียมโหลด และตัวอย่างคอลัมน์การเงิน

**จุดสังเกต:** ข้อมูลยังอยู่ในไฟล์ Parquet ของ Data Lake ยังไม่ได้โหลดเข้า Warehouse

In [ ]:
#@title 04 — แปลง Raw เป็นข้อมูล Standardized
stage = run_batch(WORKSPACE, "initial", stop_after="transform")
standard = WORKSPACE / "lake" / "standardized" / "initial"
sales_std = pd.read_parquet(standard / "Fact_Sales_Order_Item.parquet")
print("Data Lake — Standardized (ข้อมูลที่แปลงแล้ว ยังไม่ใช่ Warehouse)")
print("จำนวนไฟล์ Parquet ที่เตรียมไว้:", len(list(standard.glob("*.parquet"))))
print("จำนวนรายการขายระดับ Order Item ที่เตรียมโหลด:", len(sales_std), "แถว")
display(sales_std[["Source_Order_Item_ID", "Net_Revenue", "COGS", "Platform_Fee_Cost"]]
    .head(5).rename(columns={
        "Source_Order_Item_ID": "รหัสรายการสินค้าในออเดอร์",
        "Net_Revenue": "ยอดขายสุทธิ (บาท)",
        "COGS": "ต้นทุนสินค้า (บาท)",
        "Platform_Fee_Cost": "ค่าธรรมเนียมแพลตฟอร์ม (บาท)"
    }))
explain_result("รายการนี้แสดงตัวเลขระดับสินค้าในออเดอร์ โดยยอดขายสุทธิหักส่วนลดที่เกี่ยวข้องแล้ว ข้อมูลถูกจัดเตรียมเป็น Parquet เพื่อโหลดเข้า ER2 ในขั้นต่อไป")


### 05 — โหลดข้อมูลเข้าคลังสำหรับวิเคราะห์ (DuckDB — ER2)

**ER2 คืออะไร:** ER2 เป็น **แบบโครงสร้าง Data Warehouse** ใน Demo นี้ใช้ DuckDB สร้างตารางตาม ER2 โดยจัดข้อมูลเป็น Fact (เหตุการณ์/ตัวเลขการขาย), Dimension (รายละเอียดประกอบ) และ Mart (ตารางสรุปสำหรับวิเคราะห์)

**กำลังทำอะไร:** นำ Standardized Parquet เข้า DuckDB และสร้างข้อมูลสรุปตามวันที่และกลุ่มสินค้า

**ผลที่ควรดู:** จำนวนตารางใน Warehouse จำนวนรายการ Fact/Dimension/Mart และตัวอย่างยอดขาย-ต้นทุน

**ต่างจาก ER1:** ER1 เก็บข้อมูลธุรกรรมต้นทาง ส่วน ER2 จัดข้อมูลเพื่อการวิเคราะห์ ไม่ใช่การคัดลอกฐานข้อมูลเดิมทั้งชุด

In [ ]:
#@title 05 — โหลดข้อมูลเข้า DuckDB Data Warehouse (ER2)
stage = run_batch(WORKSPACE, "initial", stop_after="load")
with duckdb.connect(str(WORKSPACE / "er2.duckdb"), read_only=True) as con:
    er2_table_count = con.execute("""SELECT COUNT(*) FROM information_schema.tables
        WHERE table_schema='main' AND table_type='BASE TABLE'""").fetchone()[0]
    dw_counts = con.execute("""SELECT 'Fact_Sales_Order_Item' AS Table_Name, COUNT(*) AS Records FROM Fact_Sales_Order_Item
      UNION ALL SELECT 'Dim_Product',COUNT(*) FROM Dim_Product
      UNION ALL SELECT 'Mart_Product_Group_Daily_Performance',COUNT(*) FROM Mart_Product_Group_Daily_Performance""").df()
    dw_example = con.execute("""SELECT Source_Order_Item_ID, Net_Revenue, COGS, Affiliate_Cost
      FROM Fact_Sales_Order_Item LIMIT 5""").df()
names = {
    "Fact_Sales_Order_Item": "Fact — รายการขายระดับสินค้าที่ซื้อ",
    "Dim_Product": "Dimension — รายละเอียดสินค้า",
    "Mart_Product_Group_Daily_Performance": "Mart — ผลประกอบการรายวันแยกกลุ่มสินค้า"
}
dw_counts["หมายถึง"] = dw_counts["Table_Name"].map(names)
print("คลังข้อมูลสำหรับวิเคราะห์: DuckDB (ออกแบบตาม ER2)")
print("จำนวนตารางจริงใน ER2:", er2_table_count, "ตาราง")
display(dw_counts.rename(columns={"Table_Name":"ชื่อตาราง", "Records":"จำนวนแถว"}))
print("ตัวอย่างรายการขายใน Data Warehouse:")
display(dw_example.rename(columns={
    "Source_Order_Item_ID": "รหัสรายการขาย",
    "Net_Revenue": "ยอดขายสุทธิ (บาท)",
    "COGS": "ต้นทุนสินค้า (บาท)",
    "Affiliate_Cost": "ต้นทุน Affiliate (บาท)"
}))
assert int(dw_counts.loc[dw_counts.Table_Name=="Fact_Sales_Order_Item","Records"].iloc[0]) == len(sales_std)
explain_result("ตอนนี้ข้อมูลจาก Standardized Parquet เข้ามาอยู่ใน Warehouse ER2 แล้ว ต่างจาก ER1 ที่เน้นเก็บธุรกรรมต้นทาง ER2 จัด Fact, Dimension และ Mart ให้พร้อมวิเคราะห์")


### 06 — ตรวจสอบข้อมูลหลัง Run 1 (Validation)

**กำลังทำอะไร:** ตรวจว่าข้อมูลขายใน SQLite (ER1) และ DuckDB (ER2) ตรงกัน เช่น จำนวนรายการ ยอดขาย และต้นทุน พร้อมตรวจย้อนกลับไปยังรายการต้นทางได้ (Lineage)

**ผลที่ควรดู:** ตรวจทั้งหมดกี่ข้อ ผ่านกี่ข้อ ข้อไหนไม่ผ่าน และตัวอย่างที่มาของข้อมูล

**หมายเหตุ:** คำว่า PASS คือผลตรวจตามเงื่อนไขจริง ไม่ใช่สถานะที่ใส่ไว้ล่วงหน้า

In [ ]:
#@title 06 — ตรวจยอดและตามที่มาของข้อมูลหลัง Run 1
initial_result = run_batch(WORKSPACE, "initial")
INITIAL_SIGNATURE = signature(WORKSPACE)
initial_checks = pd.read_csv(WORKSPACE / "reports" / "functional_initial.csv")
print("ผลการนำเข้ารอบแรก:", initial_result["status"])
print("ตรวจความถูกต้องทั้งหมด:", len(initial_checks), "ข้อ")
print("ผ่าน:", int(initial_checks["PASS"].sum()), "| ไม่ผ่าน:",
      int((~initial_checks["PASS"]).sum()))
print("ตัวอย่างผลตรวจ ER1 เทียบ ER2:")
display(initial_checks[["Check", "Actual", "Expected", "PASS"]].head(8)
    .rename(columns={"Check":"สิ่งที่ตรวจ", "Actual":"ค่าที่ระบบคำนวณได้",
                     "Expected":"ค่าที่ควรได้", "PASS":"ผ่านหรือไม่"}))
lineage_initial = pd.read_csv(WORKSPACE / "reports" / "lineage_initial.csv")
print("ตัวอย่างการตามข้อมูลจาก Source ถึง Warehouse:")
display(lineage_initial[["Source_Order_Item_ID", "Source",
    "ER1_Net_Revenue", "ER2_Net_Revenue"]].head(3).rename(columns={
    "Source_Order_Item_ID":"รหัสรายการขาย", "Source":"ไฟล์ต้นทาง",
    "ER1_Net_Revenue":"ยอดสุทธิใน ER1", "ER2_Net_Revenue":"ยอดสุทธิใน ER2"
}))
print("จำนวนรายการขายหลัง Run 1:", INITIAL_SIGNATURE["orders"])
print("ยอดขายสุทธิหลัง Run 1:", f'{INITIAL_SIGNATURE["sales"]:,.2f}', "บาท")
assert initial_result["status"] == "SUCCESS" and not initial_result["functional_fails"]
explain_result("การโหลดข้อมูลสำเร็จยังไม่พอ ต้องตรวจยอด ER1 กับ ER2 และตามกลับไปหาไฟล์ต้นทางได้ด้วย ผลตรวจแต่ละข้อแสดงจากข้อมูลที่รันจริง")


## RUN 2 — รับข้อมูลใหม่ในฐานข้อมูลเดิม (Incremental Load)

รอบนี้ **ไม่สร้างฐานข้อมูลใหม่และไม่ลบข้อมูล Run 1** แต่รับไฟล์ CSV อีกชุดซึ่งมีทั้งคำสั่งซื้อใหม่และข้อมูลเดิมที่เปลี่ยนแปลง แล้วส่งผ่าน **ER1 → Data Lake → ER2** เหมือนรอบแรก

**ต้องแยกให้ออก:** การเพิ่ม Batch ใหม่ (Incremental) เป็นคนละอย่างกับการกด Run all ซ้ำเพื่อเริ่ม Demo ใหม่

### 07 — เปิดไฟล์ข้อมูลที่เข้ามารอบใหม่

**กำลังทำอะไร:** เปรียบเทียบ Order ID ในไฟล์ Incremental กับข้อมูลตั้งต้น แยกเป็น **ออเดอร์ใหม่** และ **ออเดอร์เดิมที่ส่งข้อมูลมาอัปเดต** เช่น สถานะชำระเงินเปลี่ยน

**ผลที่ควรดู:** จำนวนออเดอร์ใหม่ ออเดอร์เดิมที่อัปเดต และตัวอย่างข้อมูล

**หมายเหตุ:** ตัวเลขในขั้นนี้นับเป็น **ออเดอร์** ไม่ใช่จำนวนรายการสินค้าต่อออเดอร์

In [ ]:
#@title 07 — ตรวจ Source Data รอบ Incremental
incremental_dir = ROOT / "source" / "incremental"
incremental_orders = pd.read_csv(incremental_dir / "Fact_Orders.csv")
initial_ids = set(source_orders["Order_ID"].astype(str))
new_orders = incremental_orders[~incremental_orders["Order_ID"].isin(initial_ids)]
changed_orders = incremental_orders[incremental_orders["Order_ID"].isin(initial_ids)]
print("✅ Incremental Orders ใหม่:", len(new_orders))
print("✅ Order เดิมที่มีการอัปเดต:", len(changed_orders))
display(incremental_orders[["Order_ID", "Current_Status", "Paid_At"]].head(6))


### 08 — เพิ่ม/อัปเดตข้อมูลใน SQLite (ER1)

**กำลังทำอะไร:** นำ Incremental CSV เข้าฐานข้อมูล ER1 ที่สร้างไว้จาก Run 1 โดยเพิ่มออเดอร์ใหม่และอัปเดตข้อมูลออเดอร์เดิม

**ผลที่ควรดู:** จำนวน Orders ทั้งหมดและ Orders ที่มีสถานะชำระเงินแล้ว หลังนำข้อมูลรอบใหม่เข้า

**สำคัญ:** นี่คือการเปลี่ยนข้อมูลใน Database เดิม ไม่ใช่การ Reset แล้วโหลดใหม่ทั้งหมด

In [ ]:
#@title 08 — อัปเดตฐานข้อมูล ER 1
stage = run_batch(WORKSPACE, "incremental", stop_after="ingest")
with sqlite3.connect(WORKSPACE / "er1.sqlite") as con:
    er1_after = pd.read_sql_query("""SELECT COUNT(*) AS Orders_Total,
      SUM(CASE WHEN Current_Status IN ('Paid','Shipped','Delivered') THEN 1 ELSE 0 END) AS Paid_Orders
      FROM Fact_Orders""", con)
display(er1_after)
print("✅ ER1 เพิ่ม/อัปเดตข้อมูลเสร็จ; ไม่มีการ Reset")


### 09 — ส่งข้อมูลที่อัปเดตแล้วเข้า Landing และ Raw

**กำลังทำอะไร:** เก็บไฟล์ Incremental ใน Landing และดึงข้อมูลล่าสุดจาก ER1 ไปเก็บเป็น Raw Parquet รอบใหม่

**ผลที่ควรดู:** จำนวนไฟล์ที่รับเข้า จำนวน Raw Parquet และจำนวน Orders ใน Raw หลังอัปเดต

**หมายเหตุ:** Raw เป็นภาพข้อมูลจากฐาน ER1 ณ รอบนี้ จึงอาจมีข้อมูลเดิมรวมอยู่ด้วย ไม่ใช่มีแต่แถวที่เพิ่มใหม่

In [ ]:
#@title 09 — สร้าง Landing และ Raw รอบใหม่
stage = run_batch(WORKSPACE, "incremental", stop_after="extract")
landing2 = WORKSPACE / "lake" / "landing" / "incremental"
raw2 = WORKSPACE / "lake" / "raw" / "incremental"
print("✅ Landing Incremental:", len(list(landing2.glob("*.csv"))), "ไฟล์")
print("✅ Raw Snapshot:", len(list(raw2.glob("*.parquet"))), "ตาราง")
print("Raw Orders ทั้งหมด:", len(pd.read_parquet(raw2/"Fact_Orders.parquet")), "แถว")


### 10 — แปลงเฉพาะรายการที่เปลี่ยน (Standardized Delta)

**กำลังทำอะไร:** ใช้ข้อมูล Raw และรายการเปลี่ยนแปลงจากรอบ Incremental เพื่อเตรียม Fact ที่ต้องเพิ่มหรืออัปเดตใน ER2

**ผลที่ควรดู:** จำนวน **Sales Order Items ที่เปลี่ยน** และตัวอย่างยอดขายสุทธิ/ต้นทุนของรายการเหล่านั้น

**ระวังการนับ:** จำนวนนี้เป็นระดับ **Order Item** จึงไม่จำเป็นต้องเท่ากับจำนวน Orders ใหม่ในขั้น 07

In [ ]:
#@title 10 — Transform ข้อมูลที่เปลี่ยน
stage = run_batch(WORKSPACE, "incremental", stop_after="transform")
std_delta = pd.read_parquet(WORKSPACE/"lake"/"standardized"/"incremental"/"Fact_Sales_Order_Item.parquet")
print("✅ Changed Sales Items:", len(std_delta), "แถว")
display(std_delta[["Source_Order_Item_ID","Net_Revenue","COGS"]].head(4))


### 11 — อัปเดต DuckDB Warehouse (ER2 เดิม)

**กำลังทำอะไร:** เพิ่มหรืออัปเดตรายการที่เปลี่ยนใน Data Warehouse เดิม และปรับข้อมูลสรุปที่ได้รับผลกระทบ

**ผลที่ควรดู:** จำนวนรายการขายและยอดขายสุทธิ **หลัง Run 1 เทียบหลัง Run 2** โดยไม่ล้างข้อมูลรอบแรก

**สำคัญ:** ยอดก่อน–หลังต้องมาจากการ Query ฐานข้อมูลจริง

In [ ]:
#@title 11 — อัปเดต Warehouse เดิม
stage = run_batch(WORKSPACE, "incremental", stop_after="load")
after_load = signature(WORKSPACE)
display(pd.DataFrame([
    {"จุดเปรียบเทียบ":"หลัง Run 1", "Sales Items":INITIAL_SIGNATURE["orders"], "Net Revenue":INITIAL_SIGNATURE["sales"]},
    {"จุดเปรียบเทียบ":"หลัง Run 2", "Sales Items":after_load["orders"], "Net Revenue":after_load["sales"]}
]))
assert after_load["orders"] > INITIAL_SIGNATURE["orders"], "Incremental ไม่เพิ่มรายการขาย"
print("✅ Warehouse Updated")


### 12 — ตรวจความถูกต้องหลังเพิ่มข้อมูล

**กำลังทำอะไร:** ตรวจยอดจาก ER1 เทียบ ER2 อีกครั้งหลัง Incremental และเปิดตัวอย่าง Lineage เพื่อยืนยันว่าข้อมูลที่สรุปย้อนกลับไปหาต้นทางได้

**ผลที่ควรดู:** จำนวนข้อที่ผ่าน/ไม่ผ่าน และตัวอย่างรายการที่ตามข้อมูลได้จาก Source ไปยัง Warehouse

In [ ]:
#@title 12 — ตรวจคุณภาพข้อมูลหลัง Run 2
incremental_result = run_batch(WORKSPACE, "incremental")
final_check = verify(WORKSPACE/"er1.sqlite", WORKSPACE/"er2.duckdb",
                     WORKSPACE/"reports", "final")
lineage_result = lineage(WORKSPACE/"er1.sqlite", WORKSPACE/"er2.duckdb",
                         "final", WORKSPACE/"lake"/"control",
                         WORKSPACE/"reports", limit=6)
print("✅ Run 2:", incremental_result["status"])
print("✅ Functional Checks:", final_check["passed"], "/", final_check["checks"])
print("✅ Lineage:", lineage_result["traced_items"], "รายการ")
display(pd.read_csv(lineage_result["file"]).head(3))


## REPLAY — ส่ง Batch เดิมซ้ำ เพื่อตรวจว่าไม่มีข้อมูลซ้ำ

**กำลังทำอะไร:** สั่งประมวลผล Incremental Batch เดิมอีกครั้ง จากนั้นเปรียบเทียบจำนวนรายการขายและยอดเงินก่อน–หลัง

**ผลที่ถูกต้อง:** จำนวนและยอดต้องเท่าเดิม เพราะข้อมูลนี้ผ่านการประมวลผลแล้ว

> Replay **ไม่ใช่** การรับ Batch ใหม่ จึงไม่ควรแสดงว่ายอดขายเพิ่ม

In [ ]:
#@title 13 — Replay Batch เดิมเพื่อตรวจ Idempotency
before_replay = signature(WORKSPACE)
replayed = run_batch(WORKSPACE, "incremental")
after_replay = signature(WORKSPACE)
display(pd.DataFrame([
    {"ช่วง":"ก่อน Replay", "Sales Items":before_replay["orders"], "Net Revenue":before_replay["sales"]},
    {"ช่วง":"หลัง Replay", "Sales Items":after_replay["orders"], "Net Revenue":after_replay["sales"]}
]))
assert before_replay == after_replay, "❌ พบข้อมูลซ้ำหลัง Replay"
print("✅ Idempotency PASS — รัน Batch เดิมซ้ำ ตัวเลขไม่เปลี่ยน")


## ERROR TEST — ลองให้ระบบผิดพลาด แล้วตรวจการกู้คืน

**กำลังทำอะไร:** จำลองปัญหาที่อาจเกิดจริง เช่น รหัสสินค้าไม่ถูกต้อง ข้อมูล Voucher ผิด หรือ Warehouse Load ล้มเหลว ระบบจะต้องตรวจพบ บอกจุดที่ผิด แก้ข้อมูล/ยกเลิกข้อผิดพลาดที่จำลอง แล้วรันต่อจากขั้นที่ค้าง

**ผลที่ควรดู:** ชื่อปัญหา ขั้นที่ตรวจพบ Error Code ขั้นที่เริ่มใหม่ และสถานะหลัง Retry

**ทดสอบแยกจากข้อมูลหลัก:** ผลขายใน ER1/ER2 ที่สร้างไว้ก่อนหน้านี้จะไม่ถูกนำไปทำเสียโดยตั้งใจ

**ข้อยกเว้น:** เคส L02 เป็นการตรวจรับ Batch เดิมซ้ำโดยไม่สร้างข้อมูลซ้ำ ไม่จำเป็นต้องล้มเหลวก่อน

In [ ]:
#@title 14 — เลือก Error Case และดูผล Recovery
ERROR_CASE = "T01" #@param ["E01","E02","T01","T02","L01","L02"]
case_names = {
    "E01":"SKU ไม่อยู่ใน Master", "E02":"ไฟล์ Source ขาด",
    "T01":"Voucher ผิดเงื่อนไข", "T02":"Campaign วันทับซ้อน",
    "L01":"Warehouse Load Fail", "L02":"รัน Batch ซ้ำ"
}
chosen = error_suite(ROOT / "output" / "test", cases=[ERROR_CASE])
result_view = chosen[["case", "stage", "detected", "retry_from", "before", "after", "passed"]].copy()
result_view.insert(1, "คำอธิบาย", result_view["case"].map(case_names))
display(result_view)
assert chosen["passed"].all(), "Error Recovery ไม่ผ่าน"
print("✅ Error Case", ERROR_CASE, ": ตรวจพบและกู้คืนสำเร็จ")


## สรุปผล — ประสิทธิภาพ รายงานยอดขาย และผลทดสอบ

**ส่วนที่ 1 ผลทางธุรกิจ:** อ่านรายงานจาก ER2 ว่าขายได้เท่าไร ต้นทุนและกำไรเป็นเท่าไร โดยตัวเลขเป็นข้อมูลจำลอง

**ส่วนที่ 2 ผลทดสอบระบบ:** สรุปว่า Run 1/Run 2 ตรวจยอดผ่านหรือไม่ รัน Batch ซ้ำแล้วข้อมูลเปลี่ยนหรือไม่ และ Error Recovery ผ่านกี่กรณี

**ส่วนที่ 3 ประสิทธิภาพ:** จับเวลาประมวลผล (Latency) และคำนวณจำนวนรายการที่ประมวลผลต่อวินาที (Throughput) จากการรันจริงบนเครื่องนี้

ส่วนนี้เป็นการรวบรวมหลักฐานและผลที่คำนวณจริง **ไม่ใช่ชั้นเก็บข้อมูลใหม่**

In [ ]:
#@title 15 — ทดสอบครบชุด และสรุป Business Results
RUN_FULL_TEST = True #@param {type:"boolean"}
if RUN_FULL_TEST:
    six = error_suite(ROOT / "output" / "test", cases=None)
    assert len(six) == 6 and six["passed"].all()
    print("✅ Error Recovery:", int(six["passed"].sum()), "/6")
    perf = performance_suite(ROOT / "output" / "performance", sizes=(100,300))
    display(perf[["initial_source_orders","latency_seconds",
                  "throughput_paid_items_per_second"]].rename(columns={
                      "initial_source_orders":"Orders",
                      "latency_seconds":"Latency (s)",
                      "throughput_paid_items_per_second":"Throughput (items/s)"}))
    monthly, groups = business_report(WORKSPACE/"er2.duckdb", WORKSPACE/"reports")
    display(monthly[["Year_Month","Net_Revenue","Contribution_Profit",
                     "Profit_Protection_Status"]].rename(columns={
                         "Year_Month":"เดือน",
                         "Net_Revenue":"ยอดขายสุทธิ",
                         "Contribution_Profit":"กำไรหลังต้นทุน",
                         "Profit_Protection_Status":"สถานะเกณฑ์กำไร"}))
    print("✅ รายงานทั้งหมดอยู่:", WORKSPACE/"reports")
    try:
        evidence = export_github_reports(ROOT)
        print("✅ ZIP รายงานสำหรับ GitHub:", evidence["zip_path"])
    except EvidenceNotReady as exc:
        print("รายงาน GitHub ยังไม่ครบ:", exc)
else:
    print("ข้าม Full Test ได้; ผล Run 1, Run 2, Replay และ Error Case ยังใช้งานได้")


## เก็บผลลัพธ์ใน Google Drive (ไม่บังคับ)

ผลทั้งหมดอยู่ใน Colab Runtime ภายใต้ `output/` ซึ่งเป็นพื้นที่ชั่วคราว หากต้องการเก็บ Database, Data Lake และรายงานไว้ใช้ต่อ จึงเลือกบันทึกสำเนา Snapshot ไปยัง Drive ของตนเอง

ค่าเริ่มต้นเป็น **ไม่เชื่อม Drive** และการกด Run all จะข้ามการเชื่อม Drive โดยอัตโนมัติ

In [ ]:
#@title 16 — บันทึกผลลง Drive (ไม่บังคับ)
SAVE_TO_DRIVE = False #@param {type:"boolean"}
if SAVE_TO_DRIVE:
    from google.colab import drive
    from pipeline.drive_storage import save_to_drive
    drive.mount("/content/drive")
    saved = save_to_drive(ROOT, Path("/content/drive/MyDrive"))
    print("✅ บันทึกลง Google Drive:", saved["path"])
else:
    print("ไม่เชื่อม Drive — ผลทั้งหมดอยู่ใน Colab Runtime")
    print("ต้องการบันทึกถาวร: ติ๊ก SAVE_TO_DRIVE แล้วรันเฉพาะ Cell นี้")
